In [8]:
import pandas as pd
import numpy as np
import random

# 주담대, 예금은행신용, 신용대출/마통, 카드론, 리볼빙, 상환
states = ["mortgage", "deposit", "credit", "card", "revolving", "cleared"]

people = {
    "borrower_id":[i for i in range(1, 6)],
    "state":[random.choice(states) for _ in range(5)]
}

df = pd.DataFrame(people)
df

,borrower_id,state
0,1,deposit
1,2,mortgage
2,3,card
3,4,card
4,5,credit


In [ ]:
"""
───────────────────
풍선효과 트래킹 대시보드용 상태 전이확률표 (합성 데이터 시뮬레이션).

이 확률은 '실측값'이 아니라 '가정값'입니다.
   차주 단위 실제 전이율(credit→card 등)은 한국크레딧뷰로(KCB)·신용정보원의
   미공개 미시데이터로만 계산 가능하므로, 여기서는
     (1) 공개 집계 통계로 '하강 방향과 상대적 크기'를 보정하고
     (2) 남는 빈칸을 합리적 가정으로 채웠습니다.
   실배포 시 이 표는 내부 CB 데이터에서 실측한 전이행렬로 대체됩니다.
   (실측 방법은 파이프라인 3단계 '전이행렬 집계'와 동일한 계산입니다.)

근거 요약 (자세한 출처는 REFERENCES.md 참조):
   [R1] 카드론 규제 → 현금서비스·리볼빙으로 이동하는 '풍선효과'가
        반복 관측됨 (여신금융협회 리볼빙 잔액 4개월 연속 증가). newspim 2026-08
   [R2] 카드론 감소분의 약 90%를 현금서비스·리볼빙이 메워 카드빚 총액은
        거의 불변 → '더 비싼 빚으로의 대체'. ftoday 2026-07
   [R3] 은행 문턱 넘지 못한 중·저신용 차주가 카드론·리볼빙·보험계약대출 등
        '은행 밖 급전창구'로 이동. fnnews 2026-06
   [R4] 대출잔고↑ 시 연체확률↑, 특히 DTI 상승 시 급증 (KCB 미시자료 연구).
        김원혁·김승현·이윤수(2020), 국제경제연구 26(2).

반영 원칙:
   · 상태는 위험도 순서: mortgage < deposit < credit < card < revolving,
     cleared(상환·이탈)는 흡수 상태.
   · 각 리스트 = [mortgage, deposit, credit, card, revolving, cleared], 합 = 1.0
   · 규제 전(prob_normal): 유지가 지배적, 하강 약함.
   · 규제 후(prob_regulated): 하강 확률↑, 상환(cleared)↓,
     특히 card→revolving 가속([R1][R2] 근거), mortgage→credit 점프↑([R3] 근거).
   · 검산: 두 표 모두 각 줄 합 1.0, 하강확률 총합 규제전 0.30 → 규제후 1.14.
"""

STATES = ["mortgage", "deposit", "credit", "card", "revolving", "cleared"]

# 규제 전: 하강 압력 낮음 (평온기)
prob_normal = {
    #             mort  depo  cred  card  revo  clear
    "mortgage":  [0.88, 0.05, 0.04, 0.00, 0.00, 0.03],
    "deposit":   [0.06, 0.80, 0.07, 0.01, 0.00, 0.06],
    "credit":    [0.00, 0.10, 0.76, 0.05, 0.01, 0.08],
    # card→revolving은 규제 전에도 0이 아님: 급전성 이동은 상시 존재 [R1]
    "card":      [0.00, 0.01, 0.10, 0.72, 0.07, 0.10],
    "revolving": [0.00, 0.00, 0.02, 0.10, 0.80, 0.08],
    "cleared":   [0.00, 0.00, 0.03, 0.00, 0.00, 0.97],  # 흡수 상태
}

# 규제 후: 하강 확률 상승 · 상환 감소 · card→revolving 가속
prob_regulated = {
    #             mort  depo  cred  card  revo  clear
    # 주담대 규제로 막힌 수요가 신용대출·마통으로 점프 [R3]
    "mortgage":  [0.62, 0.06, 0.26, 0.02, 0.00, 0.04],
    "deposit":   [0.03, 0.62, 0.26, 0.04, 0.01, 0.04],
    # credit→card 하강 확대 [R3]
    "credit":    [0.00, 0.05, 0.66, 0.18, 0.05, 0.06],
    # card→revolving 대폭 상승: 카드론 감소분을 리볼빙이 메움 [R1][R2]
    "card":      [0.00, 0.01, 0.05, 0.62, 0.26, 0.06],
    "revolving": [0.00, 0.00, 0.01, 0.06, 0.87, 0.06],
    "cleared":   [0.00, 0.00, 0.04, 0.00, 0.00, 0.96],  # 상환 유입 소폭 감소
}


def validate(table, name=""):
    """각 줄 확률 합이 1.0인지 검산. 어긋나면 AssertionError."""
    for state, probs in table.items():
        total = round(sum(probs), 10)
        assert abs(total - 1.0) < 1e-9, f"[{name}] '{state}' 합={total} ≠ 1.0"
    return True


if __name__ == "__main__":
    validate(prob_normal, "prob_normal")
    validate(prob_regulated, "prob_regulated")
    print("확률표 검산 통과: 모든 줄 합 = 1.0")

확률표 검산 통과: 모든 줄 합 = 1.0


In [16]:
def next_state (state, regulated=False):
    """다음 달의 상태가 어떨지"""
    prob = prob_normal
    if regulated:
        prob = prob_regulated

    return np.random.choice(states, p=prob[state])

from collections import Counter

result = [next_state("credit", regulated=False) for _ in range(1000)]
print(Counter(result))

Counter({'credit': 750, 'deposit': 103, 'cleared': 73, 'card': 62, 'revolving': 12})


In [17]:
def simulate_path(start_state, reg_month, n_month):
    path = [start_state]
    for month in range(1, n_month):
        prev_state = path[-1]
        path.append(next_state(prev_state, regulated= (month >= reg_month)))
    return path

In [ ]:
from collections import Counter

paths = [simulate_path("mortgage", reg_month=6, n_month=12) for _ in range(1000)]
print("규제 전 (month 5):", Counter(p[5] for p in paths))
print("규제 후 (month 11):", Counter(p[11] for p in paths))

규제 전 (month 5): Counter({'mortgage': 571, 'cleared': 157, 'deposit': 140, 'credit': 116, 'card': 14, 'revolving': 2})
규제 후 (month 11): Counter({'cleared': 308, 'credit': 222, 'revolving': 182, 'card': 178, 'deposit': 72, 'mortgage': 38})


In [22]:
def build_panel(n_borrowers=5, reg_month=6, n_months=12):
    rows = []                              # 행들을 모을 빈 리스트
    for bid in range(n_borrowers):         # 차주 한 명씩
        start = random.choice(states) # 이 차주의 시작 상태를 정한다
        path = simulate_path(start, reg_month, n_months)  # 12개월 경로
        for month, state in enumerate(path):   # 경로를 월별로 풀어서
            rows.append({"borrower_id": bid, "month": month, "state": state})             # 한 행씩 추가
    return pd.DataFrame(rows)

In [23]:
panel = build_panel()
print(panel.head(15))
print("shape:", panel.shape)

    borrower_id  month     state
0             0      0  mortgage
1             0      1  mortgage
2             0      2  mortgage
3             0      3  mortgage
4             0      4  mortgage
5             0      5  mortgage
6             0      6  mortgage
7             0      7    credit
8             0      8    credit
9             0      9   cleared
10            0     10   cleared
11            0     11   cleared
12            1      0  mortgage
13            1      1  mortgage
14            1      2    credit
shape: (60, 3)


In [ ]:
panel["state_next"] = panel.groupby("borrower_id")['state'].shift(-1)
print(panel.head(14))

,borrower_id,month,state,state_next
0,0,0,mortgage,mortgage
1,0,1,mortgage,mortgage
2,0,2,mortgage,mortgage
3,0,3,mortgage,mortgage
4,0,4,mortgage,mortgage
5,0,5,mortgage,mortgage
6,0,6,mortgage,credit
7,0,7,credit,credit
8,0,8,credit,cleared
9,0,9,cleared,cleared


In [25]:
matrix = pd.crosstab(panel['state'], panel['state_next'])
print(matrix)

state_next  card  cleared  credit  deposit  mortgage  revolving
state                                                          
card           1        0       1        0         0          1
cleared        0        6       1        0         0          0
credit         1        2       8        1         0          2
deposit        0        1       2        9         0          0
mortgage       0        0       2        0         7          0
revolving      1        2       0        0         0          7


In [27]:
matrix_prob = pd.crosstab(panel['state'], panel['state_next'], normalize="index")
print(matrix_prob.round(2))

state_next  card  cleared  credit  deposit  mortgage  revolving
state                                                          
card        0.33     0.00    0.33     0.00      0.00       0.33
cleared     0.00     0.86    0.14     0.00      0.00       0.00
credit      0.07     0.14    0.57     0.07      0.00       0.14
deposit     0.00     0.08    0.17     0.75      0.00       0.00
mortgage    0.00     0.00    0.22     0.00      0.78       0.00
revolving   0.10     0.20    0.00     0.00      0.00       0.70


In [28]:
panel = build_panel(n_borrowers=6000, reg_month=6, n_months=12)
panel["state_next"] = panel.groupby("borrower_id")["state"].shift(-1)
print("shape:", panel.shape)   # (72000, 4) 나와야 함: 6000명 × 12개월

shape: (72000, 4)


In [29]:
# 규제 전 전이만: 출발 달(month)이 reg_month 미만
before = panel[panel["month"] < 6]
mat_before = pd.crosstab(before["state"], before["state_next"], normalize="index")

# 규제 후 전이만: 출발 달이 reg_month 이상
after = panel[panel["month"] >= 6]
mat_after = pd.crosstab(after["state"], after["state_next"], normalize="index")

print("=== 규제 전 (추정) ===")
print(mat_before.round(2))
print("\n=== 규제 후 (추정) ===")
print(mat_after.round(2))

=== 규제 전 (추정) ===
state_next  card  cleared  credit  deposit  mortgage  revolving
state                                                          
card        0.70     0.10    0.10     0.01      0.00       0.10
cleared     0.00     0.97    0.03     0.00      0.00       0.00
credit      0.07     0.08    0.75     0.09      0.00       0.02
deposit     0.01     0.06    0.10     0.77      0.06       0.00
mortgage    0.00     0.03    0.07     0.05      0.84       0.00
revolving   0.09     0.08    0.02     0.00      0.00       0.81

=== 규제 후 (추정) ===
state_next  card  cleared  credit  deposit  mortgage  revolving
state                                                          
card        0.61     0.06    0.05     0.01      0.00       0.27
cleared     0.00     0.96    0.04     0.00      0.00       0.00
credit      0.17     0.06    0.66     0.05      0.00       0.05
deposit     0.04     0.04    0.26     0.62      0.04       0.01
mortgage    0.02     0.04    0.28     0.06      0.60       0.00
rev

In [31]:
from simulation import build_panel

panel = build_panel(n_borrowers=100)   # 작게 100명으로 빠르게 테스트
print(panel.shape)   # (1200, 3) 나오면 임포트 성공

(1200, 3)
